In [65]:
import os
# #df_conflitos_score
#df_acionamentos_enriquecido_limpo, df_acion_semFaixa, df_acion_semDescricao, df_acion_semOrigem 
df_PAYJOY.to_csv('df_PAYJOY.csv', sep=';', index=False, decimal=',', encoding='utf-8-sig')
os.startfile('df_PAYJOY.csv')  # abre com o programa padrão

In [128]:
import pandas as pd
from dotenv import load_dotenv
from src.db_connection import get_connection

conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
sql_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\Daily_PayJoy.sql"

def df_consulta(conn_bd2, sql_file):
    # Lê o conteúdo da query
    with open(sql_file, "r", encoding="utf-8") as f:
        query = f.read()

    cursor = conn_bd2.cursor()
    
    try:
        # Executa a query completa
        cursor.execute(query)
        
        # Percorre todos os resultados até encontrar um com dados
        columns = None
        data = None
        
        # Tenta buscar o primeiro resultado
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            data = cursor.fetchall()
        
        # Avança pelos resultados até encontrar o último com dados
        while cursor.nextset():
            if cursor.description:
                columns = [column[0] for column in cursor.description]
                data = cursor.fetchall()
        
        # Verifica se encontrou dados
        if columns is None or data is None:
            raise ValueError("Nenhum resultado encontrado na query")
        
        # Cria o DataFrame
        df = pd.DataFrame.from_records(data, columns=columns)
        
        return df
        
    except Exception as e:
        print(f"Erro ao executar query: {e}")
        raise
    finally:
        cursor.close()

# Executa a consulta
df_consulta_df = df_consulta(conn_bd2, sql_file)

# Verifica quantas datas únicas existem
datas_unicas = df_consulta_df['DATA'].unique()
print(f"Datas encontradas: {datas_unicas}")
print(f"Total de datas: {len(datas_unicas)}")

# Define automaticamente as colunas de indicadores
# Todas as colunas EXCETO 'DATA' e 'FAIXA'
colunas_todas = df_consulta_df.columns.tolist()
colunas_id = ['DATA', 'FAIXA']  # Colunas identificadoras
value_columns = [col for col in colunas_todas if col not in colunas_id]

print(f"\nColunas identificadoras: {colunas_id}")
print(f"Total de indicadores encontrados: {len(value_columns)}")

# Faz o unpivot (melt) mantendo DATA e FAIXA como identificadores
df_transposto = df_consulta_df.melt(
    id_vars=['DATA', 'FAIXA'],
    value_vars=value_columns,
    var_name='Indicador',
    value_name='Valor'
)

print(f"\nDataFrame após melt:")
print(df_transposto.head(10))

# Pivota para colocar as datas como colunas
df_final = df_transposto.pivot_table(
    index=['FAIXA', 'Indicador'],
    columns='DATA',
    values='Valor',
    aggfunc='first'  # Usa o primeiro valor caso haja duplicatas
).reset_index()

# Remove o nome do índice das colunas (fica mais limpo)
df_final.columns.name = None

# Ordena por FAIXA e Indicador
df_final = df_final.sort_values(['FAIXA', 'Indicador']).reset_index(drop=True)

# Visualiza o resultado
print(f"\n{'='*60}")
print(f"Estrutura final:")
print(f"Total de linhas: {len(df_final)}")
print(f"Faixas únicas: {df_final['FAIXA'].unique()}")
print(f"Colunas: {df_final.columns.tolist()}")
print(f"{'='*60}")
print("\nPrimeiras 30 linhas:")
print(df_final.head(30))

df_final

Datas encontradas: [datetime.date(2025, 12, 4)]
Total de datas: 1

Colunas identificadoras: ['DATA', 'FAIXA']
Total de indicadores encontrados: 23

DataFrame após melt:
         DATA       FAIXA            Indicador  Valor
0  2025-12-04   DPD 16-30   Assigned_Portfolio   1614
1  2025-12-04   DPD 31-60   Assigned_Portfolio   2680
2  2025-12-04   DPD 61-90   Assigned_Portfolio   2084
3  2025-12-04  DPD 91-120   Assigned_Portfolio   1869
4  2025-12-04   DPD 16-30  Reachable_Portfolio      0
5  2025-12-04   DPD 31-60  Reachable_Portfolio      0
6  2025-12-04   DPD 61-90  Reachable_Portfolio      0
7  2025-12-04  DPD 91-120  Reachable_Portfolio      0
8  2025-12-04   DPD 16-30  Contacted_Portfolio   1423
9  2025-12-04   DPD 31-60  Contacted_Portfolio   2369

Estrutura final:
Total de linhas: 92
Faixas únicas: ['DPD 16-30' 'DPD 31-60' 'DPD 61-90' 'DPD 91-120']
Colunas: ['FAIXA', 'Indicador', datetime.date(2025, 12, 4)]

Primeiras 30 linhas:
        FAIXA                                  Indi

,FAIXA,Indicador,2025-12-04
0,DPD 16-30,AHT_Excluding_Short_Calls,2110
1,DPD 16-30,Active_Agents,4
2,DPD 16-30,Assigned_Portfolio,1614
3,DPD 16-30,Contacted_Portfolio,1423
4,DPD 16-30,Debt_Not_Recognized,0
...,...,...,...
87,DPD 91-120,Total_Call_Attempts,3293
88,DPD 91-120,Total_RPCs_Right_Party_Contacts,0
89,DPD 91-120,Unique_Customers_Reached,68
90,DPD 91-120,Unique_Customers_Reached_Excl_Short_Calls,47


In [12]:
import pandas as pd
from dotenv import load_dotenv
from src.db_connection import get_connection
from datetime import datetime, timedelta
import os
import sys
import re

conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
sql_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\Daily_PayJoy_prod.sql"
xlsx_file = r"\\trc-dc-ad\Planejamento\MIS\CARTEIRAS\PayJoy\base_PAYJOY.xlsx"

def identificar_maior_data_xlsx(xlsx_path):
    """
    Identifica a maior data presente nas colunas do arquivo Excel.
    Encerra a execução se o arquivo não existir.
    """
    if not os.path.exists(xlsx_path):
        print(f"\n{'='*80}")
        print(f"ERRO CRÍTICO: Arquivo não encontrado!")
        print(f"Caminho: {xlsx_path}")
        print(f"{'='*80}")
        print("\nEncerrando execução...")
        sys.exit(1)
    
    try:
        df_temp = pd.read_excel(xlsx_path, nrows=5)
        print(f"Arquivo Excel lido com sucesso!")
        
        colunas = df_temp.columns.tolist()
        print(f"\nColunas encontradas no arquivo: {colunas}")
        
        ultima_data = None
        
        for col in reversed(colunas):
            if col in ['FAIXA', 'Indicador']:
                continue
            
            try:
                if isinstance(col, (pd.Timestamp, datetime)):
                    if isinstance(col, pd.Timestamp):
                        ultima_data = col.to_pydatetime()
                    else:
                        ultima_data = col
                    print(f"\nMaior data encontrada: {ultima_data.strftime('%Y-%m-%d')} (coluna já era datetime)")
                    return ultima_data
                
                col_limpo = str(col).strip()
                
                for formato in ['%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d', '%d-%m-%Y']:
                    try:
                        data = datetime.strptime(col_limpo, formato)
                        ultima_data = data
                        print(f"\nMaior data encontrada: {ultima_data.strftime('%Y-%m-%d')} (coluna: '{col}')")
                        return ultima_data
                    except:
                        continue
            except Exception as e:
                continue
        
        if ultima_data is None:
            print("\n" + "="*80)
            print("ERRO: Nenhuma data válida encontrada nas colunas do arquivo!")
            print("Colunas analisadas:", [col for col in colunas if col not in ['FAIXA', 'Indicador']])
            print("="*80)
            print("\nEncerrando execução...")
            sys.exit(1)
        
        return ultima_data
            
    except Exception as e:
        print(f"\n{'='*80}")
        print(f"ERRO ao ler arquivo Excel: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        print("\nEncerrando execução...")
        sys.exit(1)

def calcular_data_inicio(maior_data_xlsx):
    """
    Calcula a data de início para a consulta (dia seguinte à última data do arquivo).
    """
    if maior_data_xlsx is None:
        print(f"\n{'='*80}")
        print("ERRO: Não foi possível identificar a data inicial!")
        print(f"{'='*80}")
        sys.exit(1)
    
    data_inicio = maior_data_xlsx.date() + timedelta(days=1)
    print(f"\nData início para consulta: {data_inicio.strftime('%Y-%m-%d')}")
    return data_inicio

def df_consulta(conn_bd2, sql_file, data_inicio):
    """
    Executa a consulta substituindo a variável @dataIni no arquivo SQL.
    
    Parâmetros:
    -----------
    conn_bd2 : pyodbc.Connection
        Conexão com o banco de dados
    sql_file : str
        Caminho do arquivo SQL
    data_inicio : datetime.date
        Data a ser substituída na variável @dataIni
    
    Retorna:
    --------
    pd.DataFrame
        DataFrame com os resultados da consulta
    """
    # Lê o conteúdo da query
    with open(sql_file, "r", encoding="utf-8") as f:
        query_original = f.read()
    
    # Formata a data no formato YYYY-MM-DD
    data_str = data_inicio.strftime('%Y-%m-%d')
    
    print(f"\n[SUBSTITUIÇÃO] Buscando padrão '@dataIni' no SQL...")
    print(f"[SUBSTITUIÇÃO] Formatando data: {data_str}")
    
    # Usa regex para encontrar e substituir de forma robusta
    # Padrão: declare @dataIni as date = 'YYYY-MM-DD' (com variações de espaço)
    padrao = r"declare\s+@dataIni\s+as\s+date\s*=\s*'[^']*'"
    
    # Verificar se encontrou o padrão
    if re.search(padrao, query_original, re.IGNORECASE):
        print(f"[✓] Padrão encontrado no SQL")
        substituicao = f"declare @dataIni as date = '{data_str}'"
        query_modificada = re.sub(padrao, substituicao, query_original, flags=re.IGNORECASE)
        
        # Log das mudanças
        linhas_original = query_original.split('\n')
        linhas_modificada = query_modificada.split('\n')
        
        for i, (orig, mod) in enumerate(zip(linhas_original[:50], linhas_modificada[:50])):
            if orig != mod:
                print(f"\n[LINHA {i+1}] Original:")
                print(f"  {orig.strip()}")
                print(f"[LINHA {i+1}] Modificada:")
                print(f"  {mod.strip()}")
    else:
        print(f"[✗] AVISO: Padrão '@dataIni' NÃO encontrado no SQL!")
        print(f"[✗] A consulta será executada sem modificação da data")
        query_modificada = query_original
    
    cursor = conn_bd2.cursor()
    
    try:
        print(f"\n[EXECUÇÃO] Executando consulta com @dataIni = '{data_str}'...")
        cursor.execute(query_modificada)
        
        # Navega pelos resultados até encontrar o último SELECT
        columns = None
        data = None
        
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            data = cursor.fetchall()
            print(f"[✓] Primeiro result set encontrado: {len(data)} linhas")
        
        result_set_count = 1
        while cursor.nextset():
            if cursor.description:
                result_set_count += 1
                columns = [column[0] for column in cursor.description]
                data = cursor.fetchall()
                print(f"[✓] Result set #{result_set_count} encontrado: {len(data)} linhas")
        
        if columns is None or data is None:
            raise ValueError("Nenhum resultado encontrado na query")
        
        df = pd.DataFrame.from_records(data, columns=columns)
        print(f"\n[✓] Query retornou {len(df)} linhas com {len(columns)} colunas")
        
        return df
        
    except Exception as e:
        print(f"\n[✗] Erro ao executar query: {e}")
        import traceback
        traceback.print_exc()
        raise
    finally:
        cursor.close()

# ============================================================================
# EXECUÇÃO PRINCIPAL
# ============================================================================

print("="*80)
print("INICIANDO PROCESSAMENTO PayJoy")
print("="*80)

# 1. Identifica a maior data no arquivo Excel
maior_data_xlsx = identificar_maior_data_xlsx(xlsx_file)

# 2. Calcula data de início (dia seguinte à última data do arquivo)
data_inicio = calcular_data_inicio(maior_data_xlsx)

# 3. Verifica se há dados para processar
ontem = datetime.now().date() - timedelta(days=1)

if data_inicio > ontem:
    print(f"\n{'='*80}")
    print("Relatório já está atualizado! Nenhuma data para processar.")
    print(f"{'='*80}")
    df_resultado = pd.DataFrame()
else:
    print(f"\nPeríodo a processar: de {data_inicio.strftime('%Y-%m-%d')} até {ontem.strftime('%Y-%m-%d')}")
    
    # 4. Executa a consulta
    df_resultado = df_consulta(conn_bd2, sql_file, data_inicio)
    
    print(f"\n{'='*80}")
    print("CONSULTA EXECUTADA COM SUCESSO!")
    print(f"{'='*80}")
    print(f"Total de linhas retornadas: {len(df_resultado)}")
    print(f"Colunas: {df_resultado.columns.tolist()}")
    print(f"{'='*80}")
    
    if len(df_resultado) > 0:
        print("\nPrimeiras 10 linhas do resultado:")
        print(df_resultado.head(10))
        
        print("\nDatas únicas no resultado:")
        if 'DATA' in df_resultado.columns:
            datas_unicas = sorted(df_resultado['DATA'].unique())
            print(datas_unicas)

conn_bd2.close()
print("\nConexão fechada.")
print(f"\nDataFrame 'df_resultado' disponível com {len(df_resultado)} linhas")

INICIANDO PROCESSAMENTO PayJoy
Arquivo Excel lido com sucesso!

Colunas encontradas no arquivo: ['FAIXA', 'Indicador', datetime.datetime(2025, 12, 4, 0, 0), datetime.datetime(2025, 12, 6, 0, 0), datetime.datetime(2025, 12, 8, 0, 0), datetime.datetime(2025, 12, 10, 0, 0), datetime.datetime(2025, 12, 11, 0, 0), datetime.datetime(2025, 12, 12, 0, 0), datetime.datetime(2025, 12, 13, 0, 0), datetime.datetime(2025, 12, 15, 0, 0)]

Maior data encontrada: 2025-12-15 (coluna já era datetime)

Data início para consulta: 2025-12-16

Período a processar: de 2025-12-16 até 2025-12-25

[SUBSTITUIÇÃO] Buscando padrão '@dataIni' no SQL...
[SUBSTITUIÇÃO] Formatando data: 2025-12-16
[✓] Padrão encontrado no SQL

[LINHA 9] Original:
  declare @dataIni as date = '2025-12-05';
[LINHA 9] Modificada:
  declare @dataIni as date = '2025-12-16';

[EXECUÇÃO] Executando consulta com @dataIni = '2025-12-16'...
[✓] Result set #2 encontrado: 28 linhas

[✓] Query retornou 28 linhas com 25 colunas

CONSULTA EXECUTADA 

In [13]:

# Verifica quantas datas únicas existem
datas_unicas = df_resultado['DATA'].unique()
print(f"Datas encontradas: {datas_unicas}")
print(f"Total de datas: {len(datas_unicas)}")

# Define automaticamente as colunas de indicadores
# Todas as colunas EXCETO 'DATA' e 'FAIXA'
colunas_todas = df_resultado.columns.tolist()
colunas_id = ['DATA', 'FAIXA']  # Colunas identificadoras
value_columns = [col for col in colunas_todas if col not in colunas_id]

print(f"\nColunas identificadoras: {colunas_id}")
print(f"Total de indicadores encontrados: {len(value_columns)}")

# Faz o unpivot (melt) mantendo DATA e FAIXA como identificadores
df_transposto = df_resultado.melt(
    id_vars=['DATA', 'FAIXA'],
    value_vars=value_columns,
    var_name='Indicador',
    value_name='Valor'
)

print(f"\nDataFrame após melt:")
print(df_transposto.head(10))

# Pivota para colocar as datas como colunas
df_final = df_transposto.pivot_table(
    index=['FAIXA', 'Indicador'],
    columns='DATA',
    values='Valor',
    aggfunc='first'  # Usa o primeiro valor caso haja duplicatas
).reset_index()

# Remove o nome do índice das colunas (fica mais limpo)
df_final.columns.name = None

# Ordena por FAIXA e Indicador
df_final = df_final.sort_values(['FAIXA', 'Indicador']).reset_index(drop=True)

# Visualiza o resultado
print(f"\n{'='*60}")
print(f"Estrutura final:")
print(f"Total de linhas: {len(df_final)}")
print(f"Faixas únicas: {df_final['FAIXA'].unique()}")
print(f"Colunas: {df_final.columns.tolist()}")
print(f"{'='*60}")
print("\nPrimeiras 30 linhas:")
print(df_final.head(30))

df_final

Datas encontradas: [datetime.date(2025, 12, 16) datetime.date(2025, 12, 17)
 datetime.date(2025, 12, 18) datetime.date(2025, 12, 19)
 datetime.date(2025, 12, 20) datetime.date(2025, 12, 22)
 datetime.date(2025, 12, 23) datetime.date(2025, 12, 24)]
Total de datas: 8

Colunas identificadoras: ['DATA', 'FAIXA']
Total de indicadores encontrados: 23

DataFrame após melt:
         DATA       FAIXA           Indicador  Valor
0  2025-12-16   DPD 61-90  Assigned_Portfolio   2035
1  2025-12-16   DPD 16-30  Assigned_Portfolio   2399
2  2025-12-16  DPD 91-120  Assigned_Portfolio   1408
3  2025-12-16   DPD 31-60  Assigned_Portfolio   2558
4  2025-12-17   DPD 61-90  Assigned_Portfolio   2035
5  2025-12-17  DPD 91-120  Assigned_Portfolio   1408
6  2025-12-17   DPD 31-60  Assigned_Portfolio   2558
7  2025-12-17   DPD 16-30  Assigned_Portfolio   2399
8  2025-12-18   DPD 31-60  Assigned_Portfolio   2558
9  2025-12-18  DPD 91-120  Assigned_Portfolio   1408

Estrutura final:
Total de linhas: 92
Faixas úni

,FAIXA,Indicador,2025-12-16,2025-12-17,2025-12-18,2025-12-19,2025-12-20,2025-12-22,2025-12-23,2025-12-24
0,DPD 16-30,AHT_Excluding_Short_Calls,2385.0,1979.0,2479.0,2500.0,1108.0,1555.0,1221.0,70.0
1,DPD 16-30,Active_Agents,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
2,DPD 16-30,Assigned_Portfolio,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0
3,DPD 16-30,Contacted_Portfolio,1234.0,2139.0,2166.0,2006.0,1351.0,2158.0,1850.0,271.0
4,DPD 16-30,Debt_Not_Recognized,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
87,DPD 91-120,Total_Call_Attempts,1477.0,1150.0,1662.0,1126.0,1321.0,819.0,NaN,NaN
88,DPD 91-120,Total_RPCs_Right_Party_Contacts,7.0,1.0,5.0,3.0,2.0,0.0,NaN,NaN
89,DPD 91-120,Unique_Customers_Reached,28.0,18.0,32.0,27.0,22.0,14.0,NaN,NaN
90,DPD 91-120,Unique_Customers_Reached_Excl_Short_Calls,21.0,12.0,19.0,12.0,10.0,5.0,NaN,NaN


In [4]:
import pandas as pd
from src.db_connection import get_connection

def df_contratos_payjoy():
    """
    Busca os contratos PayJoy com suas respectivas faixas (DPD_BUCKET)
    
    Returns:
        DataFrame com colunas: Assigned_Portfolio, FAIXA
    """
    # Cria a conexão
    conn_bd2 = get_connection("SERVER_BD2", "DATABASE_BD2")
    
    # Query SQL
    query = """
    SELECT
        Assigned_Portfolio
    ,   DPD_BUCKET as FAIXA
    FROM OPENQUERY([TRC_BD_LINKED],
    '
        SELECT 
            ltrim(rtrim(contrato_fin)) Assigned_Portfolio
        ,   DPD_BUCKET
        ,   cast(CALENDAR_DATE as date) dt_base 
        FROM SRC..AUX_PAYJOY_REMESSA 
        WHERE CALENDAR_DATE = (SELECT MAX(CALENDAR_DATE) FROM SRC..AUX_PAYJOY_REMESSA)
    '
    )
    """
    
    try:
        # Executa a query e retorna o DataFrame
        df = pd.read_sql(query, conn_bd2)
        
        print(f"Total de contratos: {len(df)}")
        print(f"Faixas disponíveis: {df['FAIXA'].unique()}")
        
        return df
        
    except Exception as e:
        print(f"Erro ao executar query: {e}")
        raise
    finally:
        conn_bd2.close()

# Executa a função
df_contratos = df_contratos_payjoy()
df_contratos

Total de contratos: 8400
Faixas disponíveis: ['DPD 61-90' 'DPD 91-120' 'DPD 16-30' 'DPD 31-60']


C:\Users\claudiano.alves\AppData\Local\Temp\ipykernel_16048\4052977116.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn_bd2)


,Assigned_Portfolio,FAIXA
0,54988427ID,DPD 61-90
1,54988428ID,DPD 91-120
2,54988429ID,DPD 91-120
3,54988430ID,DPD 16-30
4,54988431ID,DPD 91-120
...,...,...
8395,55154585ID,DPD 16-30
8396,55154586ID,DPD 16-30
8397,55154587ID,DPD 16-30
8398,55154588ID,DPD 16-30


In [2]:
import pandas as pd
import os
from glob import glob
import re

def consolidar_mailings_payjoy(caminho_pasta=r"\\trc-dc-ad\Planejamento\00 - USUÁRIOS\0003_Daniel Kodama\PAYJOY\mailings payjoy"):
    """
    Consolida todos os arquivos CSV de mailings PayJoy em um único DataFrame
    """
    
    arquivos = glob(os.path.join(caminho_pasta, "*.csv"))
    
    if not arquivos:
        print(f"Nenhum arquivo CSV encontrado em: {caminho_pasta}")
        return pd.DataFrame()
    
    print(f"Encontrados {len(arquivos)} arquivos CSV\n")
    
    lista_dfs = []
    arquivos_com_erro = []
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            nome_sem_extensao = nome_arquivo.replace('.csv', '')
            
            # Busca padrão de data no nome: DD.MM ou DD/MM
            match = re.search(r'(\d{1,2})[./](\d{2})', nome_sem_extensao)
            
            if match:
                dia = match.group(1).zfill(2)  # Adiciona zero à esquerda se necessário
                mes = match.group(2)
                data_str = f"{dia}/{mes}"
            else:
                # Se não encontrar data, usa os últimos 4 caracteres (método antigo)
                ultimos_4_chars = nome_sem_extensao[-4:]
                data_str = ultimos_4_chars.replace('.', '/')
            
            # Tenta diferentes combinações de encoding e separador
            df_temp = None
            for sep in [';', ',', '\t']:
                for enc in ['latin-1', 'utf-8', 'cp1252']:
                    try:
                        df_temp = pd.read_csv(arquivo, usecols=['CONTRATO'], encoding=enc, sep=sep)
                        if not df_temp.empty and 'CONTRATO' in df_temp.columns:
                            break
                    except:
                        continue
                if df_temp is not None and not df_temp.empty:
                    break
            
            if df_temp is None or df_temp.empty:
                raise Exception("Não foi possível ler o arquivo com nenhuma combinação")
            
            df_temp['DATA'] = data_str
            lista_dfs.append(df_temp)
            
            print(f"✓ {nome_arquivo} - {len(df_temp)} registros - Data: {data_str}")
            
        except Exception as e:
            print(f"✗ ERRO em {nome_arquivo}: {e}")
            arquivos_com_erro.append((nome_arquivo, str(e)))
            continue
    
    if arquivos_com_erro:
        print(f"\n{'='*60}")
        print("ARQUIVOS COM ERRO:")
        for nome, erro in arquivos_com_erro:
            print(f"  - {nome}: {erro}")
        print(f"{'='*60}\n")
    
    if lista_dfs:
        df_consolidado = pd.concat(lista_dfs, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"Total de registros consolidados: {len(df_consolidado)}")
        print(f"Datas únicas: {sorted(df_consolidado['DATA'].unique())}")
        print(f"Total de contratos únicos: {df_consolidado['CONTRATO'].nunique()}")
        print(f"{'='*60}")
        
        return df_consolidado
    else:
        print("Nenhum dado foi consolidado.")
        return pd.DataFrame()

df_mailings = consolidar_mailings_payjoy()
df_mailings

Encontrados 24 arquivos CSV

✗ ERRO em PAYJOY-ATIVO.01.12.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.02.12.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.03.12.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.18.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.19.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.21.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.24.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.25.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.26.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.27.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ ERRO em PAYJOY-ATIVO.28.11.csv: Não foi possível ler o arquivo com nenhuma combinação
✗ E

,CONTRATO,DATA
0,55154266ID,22/12
1,55153637ID,22/12
2,55154323ID,22/12
3,55154458ID,22/12
4,55154329ID,22/12
...,...,...
8394,54991533ID,22/12
8395,54991324ID,22/12
8396,54993200ID,22/12
8397,54995207ID,22/12


In [14]:
def consolidar_mailings_por_faixa(df_mailings, df_contratos, ordenar_datas=True, contratos_unicos=True):
    """
    Cruza os mailings com os contratos para obter faixas e gera tabela pivotada
    """
    
    # LIMPEZA: Remove espaços em branco das colunas de chave
    df_mailings['CONTRATO'] = df_mailings['CONTRATO'].astype(str).str.strip()
    df_contratos['Assigned_Portfolio'] = df_contratos['Assigned_Portfolio'].astype(str).str.strip()
    
    # Cruzamento
    df_mailings_com_faixa = df_mailings.merge(
        df_contratos,
        left_on='CONTRATO',
        right_on='Assigned_Portfolio',
        how='left'
    )
    
    # Contagem
    if contratos_unicos:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])['CONTRATO'].nunique().reset_index()
    else:
        df_contagem = df_mailings_com_faixa.groupby(['FAIXA', 'DATA'])['CONTRATO'].count().reset_index()
    
    df_contagem.columns = ['FAIXA', 'DATA', 'CONTAGEM']
    
    # Pivota
    df_pivot = df_contagem.pivot(
        index='FAIXA',
        columns='DATA',
        values='CONTAGEM'
    ).fillna(0).astype(int)
    
    # Ordena datas se solicitado
    if ordenar_datas:
        def ordenar_data(data_str):
            """Converte DD/MM para tupla (dia, mes) para ordenação"""
            try:
                dia, mes = data_str.split('/')
                return (int(mes), int(dia))
            except:
                return (99, 99)
        
        colunas_ordenadas = sorted(df_pivot.columns, key=ordenar_data)
        df_pivot = df_pivot[colunas_ordenadas]
    
    # Reset index
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

# Uso:
df_pivot = consolidar_mailings_por_faixa(df_mailings, df_contratos)
df_pivot

DATA,FAIXA,22/12
0,DPD 16-30,2399
1,DPD 31-60,2557
2,DPD 61-90,2035
3,DPD 91-120,1408


In [15]:
import pandas as pd

def substituir_zeros_com_pivot(df_transposto, df_pivot):
    """
    Substitui valores 0 do df_transposto pelos valores do df_pivot,
    respeitando FAIXA + data.
    
    Args:
        df_transposto: DataFrame com estrutura [FAIXA, Indicador, 2025-12-04, 2025-12-05, ...]
        df_pivot: DataFrame com estrutura [FAIXA, 3/12, 4/12, 5/12, ...]
    
    Returns:
        DataFrame atualizado
    """
    
    df_resultado = df_transposto.copy()
    
    # Pegar as colunas de data do df_transposto
    colunas_data = [col for col in df_transposto.columns if col not in ['FAIXA', 'Indicador']]
    
    # Para cada linha do df_pivot
    for _, row_pivot in df_pivot.iterrows():
        faixa = row_pivot["FAIXA"]
        
        # Para cada coluna de data no df_transposto
        for col_transposto in colunas_data:
            # Converter coluna do df_transposto para formato do df_pivot
            if isinstance(col_transposto, str) and '-' in col_transposto:
                partes = col_transposto.split('-')
                dia = partes[2].lstrip('0') or '0'
                mes = partes[1].lstrip('0') or '0'
                col_pivot_format = f"{dia}/{mes}"
            elif hasattr(col_transposto, 'day') and hasattr(col_transposto, 'month'):
                dia = str(col_transposto.day)
                mes = str(col_transposto.month)
                col_pivot_format = f"{dia}/{mes}"
            else:
                continue
            
            # Se essa coluna existe no df_pivot
            if col_pivot_format in df_pivot.columns:
                valor_pivot = row_pivot[col_pivot_format]
                
                # Substituir os zeros APENAS para o indicador Reachable_Portfolio
                mask = (
                    (df_resultado["FAIXA"] == faixa) & 
                    (df_resultado["Indicador"] == "Reachable_Portfolio") &
                    (df_resultado[col_transposto] == 0)
                )
                df_resultado.loc[mask, col_transposto] = valor_pivot
    
    return df_resultado


df_PAYJOY = substituir_zeros_com_pivot(df_final, df_pivot)
print("DataFrame atualizado:")
df_PAYJOY

DataFrame atualizado:


,FAIXA,Indicador,2025-12-16,2025-12-17,2025-12-18,2025-12-19,2025-12-20,2025-12-22,2025-12-23,2025-12-24
0,DPD 16-30,AHT_Excluding_Short_Calls,2385.0,1979.0,2479.0,2500.0,1108.0,1555.0,1221.0,70.0
1,DPD 16-30,Active_Agents,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
2,DPD 16-30,Assigned_Portfolio,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0,2399.0
3,DPD 16-30,Contacted_Portfolio,1234.0,2139.0,2166.0,2006.0,1351.0,2158.0,1850.0,271.0
4,DPD 16-30,Debt_Not_Recognized,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
87,DPD 91-120,Total_Call_Attempts,1477.0,1150.0,1662.0,1126.0,1321.0,819.0,NaN,NaN
88,DPD 91-120,Total_RPCs_Right_Party_Contacts,7.0,1.0,5.0,3.0,2.0,0.0,NaN,NaN
89,DPD 91-120,Unique_Customers_Reached,28.0,18.0,32.0,27.0,22.0,14.0,NaN,NaN
90,DPD 91-120,Unique_Customers_Reached_Excl_Short_Calls,21.0,12.0,19.0,12.0,10.0,5.0,NaN,NaN


In [16]:
df_transposto

,DATA,FAIXA,Indicador,Valor
0,2025-12-16,DPD 61-90,Assigned_Portfolio,2035
1,2025-12-16,DPD 16-30,Assigned_Portfolio,2399
2,2025-12-16,DPD 91-120,Assigned_Portfolio,1408
3,2025-12-16,DPD 31-60,Assigned_Portfolio,2558
4,2025-12-17,DPD 61-90,Assigned_Portfolio,2035
...,...,...,...,...
639,2025-12-22,DPD 91-120,Number_Of_Email_sent,0
640,2025-12-23,DPD 16-30,Number_Of_Email_sent,0
641,2025-12-23,DPD 31-60,Number_Of_Email_sent,0
642,2025-12-24,DPD 16-30,Number_Of_Email_sent,0


In [17]:
import pandas as pd

# Exemplo: supondo que a coluna que guarda o nome do indicador se chama "Indicador"
df_filtrado = df_PAYJOY[df_PAYJOY["Indicador"] == "Reachable_Portfolio"]
df_filtrado

,FAIXA,Indicador,2025-12-16,2025-12-17,2025-12-18,2025-12-19,2025-12-20,2025-12-22,2025-12-23,2025-12-24
14,DPD 16-30,Reachable_Portfolio,0.0,0.0,0.0,0.0,0.0,2399.0,0.0,0.0
37,DPD 31-60,Reachable_Portfolio,0.0,0.0,0.0,0.0,0.0,2557.0,0.0,NaN
60,DPD 61-90,Reachable_Portfolio,0.0,0.0,0.0,0.0,0.0,2035.0,NaN,0.0
83,DPD 91-120,Reachable_Portfolio,0.0,0.0,0.0,0.0,0.0,1408.0,NaN,NaN
